# OPRB Test-Set Evaluation Template (Reusable Across Models)

Reusable zero-shot evaluation pipeline for **any** pretrained lightweight VLM or OCR engine on the **OPRB test set** (9,000 images: 6,000 `text` + 3,000 `nontext`).

**How to use this template for each model:**
1. Edit **Section 3 (Configuration)**: set `MODEL_TAG` and `MODEL_ID` (and `MAX_SAMPLES_PER_CATEGORY = None` for the full 9,000-image run).
2. Uncomment & fill in **Section 1** (extra `pip install` packages for your engine, if any).
3. Uncomment & fill in **Section 7** (load your VLM or OCR engine).
4. Uncomment & fill in **Section 8** (`transcribe_page(image_path)` returning `(text, elapsed_sec, generated_tokens, possibly_truncated)`).
5. Run all cells — GT loading, text normalization, metrics, Drive checkpointing, plots, and `.zip` export are identical across all models for fair comparison.

## 1. Install dependencies

In [ ]:
%pip -q install "huggingface_hub>=0.30" "jiwer>=3.0" "pandas>=2.0" matplotlib Pillow tqdm

# --- FILL IN: Engine-specific packages (uncomment what your model needs) ---
# For HuggingFace VLMs:
# %pip -q install "transformers>=4.53,<5" "accelerate>=1.0" "bitsandbytes>=0.45"

# For OCR engines (examples — pick one):
# %pip -q install easyocr
# %pip -q install paddlepaddle-gpu paddleocr
# %pip -q install rapidocr-onnxruntime
# %pip -q install surya-ocr
# !apt-get -qq install -y tesseract-ocr && pip -q install pytesseract

## 2. Imports, Google Drive mount, and hardware check

In [ ]:
import json, random, re, shutil, tarfile, time, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from huggingface_hub import hf_hub_download
from jiwer import cer, wer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ImportError:
    DRIVE_ROOT = Path(".")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Device:", DEVICE, "|", GPU_NAME)
print("Torch:", torch.__version__)

## 3. Configuration — edit per model

In [ ]:
REPO_ID = "kpurkayastha/OPRB"

# --- FILL IN: Unique tag & identifier for this model ---
MODEL_TAG = "my_model_tag"          # e.g. "paddleocr", "easyocr", "smolvlm_2b", "internvl2_5_2b"
MODEL_ID = "org/model-or-engine"    # e.g. "HuggingFaceTB/SmolVLM-Instruct" or "EasyOCR-en"
MODEL_CATEGORY = "OCR"              # "VLM" or "OCR" (for your summary/config)
PRECISION_NOTE = "FP16"             # e.g. "bfloat16", "4-bit NF4", "FP32", "ONNX"

DATA_ROOT = Path("/content/oprb_test")
OUTPUT_DIR = DRIVE_ROOT / "oprb_results" / MODEL_TAG

CATEGORIES = ["text", "nontext"]
MAX_SAMPLES_PER_CATEGORY = None     # Evaluates ALL 9,000 test images (6,000 text + 3,000 nontext)
RANDOM_SEED = 42
NORMALIZE_CASE = True               # Keep True across all models for fair comparison
SAVE_EVERY = 20                     # Save checkpoint CSV every N images (use 1 for slow VLMs, 20-50 for fast OCR)

# Optional VLM settings (unused by classic OCR engines)
MAX_IMAGE_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 2048

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_ID, "| Output:", OUTPUT_DIR)
print("Max samples per category:", MAX_SAMPLES_PER_CATEGORY)

## 4. Download and extract OPRB test images and GT text

In [ ]:
def extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:
    destination.mkdir(parents=True, exist_ok=True)
    count = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in tqdm(archive, desc=f"Extract {destination.parent.name}/{destination.name}", leave=False):
            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):
                continue
            target = destination / Path(member.name).name
            if not target.exists():
                stream = archive.extractfile(member)
                if stream is None:
                    continue
                with stream, target.open("wb") as out:
                    shutil.copyfileobj(stream, out)
            count += 1
    return count


def download_test_category(category: str):
    for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):
        repo_path = f"test/{category}/{kind}.tar.gz"
        target_dir = DATA_ROOT / category / kind
        existing = list(target_dir.glob(f"*{suffix}")) if target_dir.exists() else []
        expected = 6000 if category == "text" else 3000
        if len(existing) >= expected:
            print(f"Reuse {category}/{kind}: {len(existing)} files")
            continue
        print(f"Downloading {repo_path} ...")
        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=repo_path)
        n = extract_flat(tar_path, target_dir, suffix)
        print(f"  {category}/{kind}: {n:,} files -> {target_dir}")


for category in CATEGORIES:
    download_test_category(category)

## 5. Pair input images with evaluation annotations

In [ ]:
def parse_occlusion_metadata(stem: str):
    parts = stem.split("__", 2)
    return {
        "occlusion_type": parts[0] if parts else "unknown",
        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
    }


def collect_test_pairs(categories):
    pairs = []
    for category in categories:
        image_dir = DATA_ROOT / category / "img"
        gt_dir = DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())
        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")
        if MAX_SAMPLES_PER_CATEGORY is not None:
            rng = random.Random(RANDOM_SEED)
            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))
        for stem in common:
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                **parse_occlusion_metadata(stem),
            })
    return pairs


pairs = collect_test_pairs(CATEGORIES)
print(f"Selected {len(pairs):,} samples.")

## 6. Ground-truth loader and shared evaluation metrics (do not change across models)

In [ ]:
def read_gt_tokens(gt_path: str):
    tokens = []
    with open(gt_path, "r", encoding="utf-8-sig", errors="replace") as file:
        for line in file:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 2 and fields[0].strip():
                tokens.append(fields[0].strip())
    return tokens


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold() if NORMALIZE_CASE else text


def bag_of_words_prf(reference: str, prediction: str):
    ref = Counter(reference.split())
    hyp = Counter(prediction.split())
    matched = sum((ref & hyp).values())
    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())
    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)
    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def compute_metrics(ground_truth: str, prediction: str):
    ref, hyp = clean_text(ground_truth), clean_text(prediction)
    if not ref:
        raise ValueError("Empty GT transcript; cannot compute meaningful CER/WER.")
    p, r, f1 = bag_of_words_prf(ref, hyp)
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }


assert compute_metrics("The cat sat", "The cat")["wer"] == 1 / 3
print("First GT tokens:", read_gt_tokens(pairs[0]["gt_path"])[:20])

## 7. Load pretrained model / OCR engine — FILL IN THIS SECTION

In [ ]:
# === OPTION A: HuggingFace VLM (uncomment & adapt) ===
# from transformers import AutoProcessor, AutoModelForVision2Seq
# processor = AutoProcessor.from_pretrained(MODEL_ID)
# model = AutoModelForVision2Seq.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# ).eval()

# === OPTION B: EasyOCR (uncomment) ===
# import easyocr
# reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

# === OPTION C: PaddleOCR (uncomment) ===
# from paddleocr import PaddleOCR
# ocr_engine = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)

# === OPTION D: RapidOCR (uncomment) ===
# from rapidocr_onnxruntime import RapidOCR
# ocr_engine = RapidOCR()

# === OPTION E: Tesseract (uncomment) ===
# import pytesseract

raise NotImplementedError("Uncomment and configure your model/OCR engine in Section 7.")

## 8. Define `transcribe_page(image_path)` and test on 1 image — FILL IN THIS SECTION

In [ ]:
OCR_PROMPT = (
    "Transcribe all text visible on this document page in natural reading order. "
    "Return only the transcription, without introductory text or explanations. "
    "Do not describe the page, and do not invent text that you cannot read."
)


def transcribe_page(image_path: str):
    """Return (text, elapsed_sec, generated_tokens, possibly_truncated)."""
    start = time.perf_counter()

    # --- FILL IN: Run your model/engine on image_path ---
    # Example (EasyOCR):
    # lines = reader.readtext(image_path, detail=0, paragraph=True)
    # text = " ".join(lines)
    # generated_tokens, possibly_truncated = 0, False

    # Example (PaddleOCR):
    # res = ocr_engine.ocr(image_path, cls=True)
    # text = " ".join(line[1][0] for page in (res or []) if page for line in page)
    # generated_tokens, possibly_truncated = 0, False

    # Example (Tesseract):
    # with Image.open(image_path) as im:
    #     text = pytesseract.image_to_string(im.convert("RGB"))
    # generated_tokens, possibly_truncated = 0, False

    raise NotImplementedError("Implement transcribe_page(image_path) in Section 8.")

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - start
    return text.strip(), elapsed_sec, generated_tokens, possibly_truncated


sample = pairs[0]
ex_pred, ex_time, ex_tok, ex_trunc = transcribe_page(sample["image_path"])
ex_gt = " ".join(read_gt_tokens(sample["gt_path"]))
print("LATENCY (s):", round(ex_time, 3))
print("GT (first 400):", ex_gt[:400])
print("PRED (first 400):", ex_pred[:400])
print("METRICS:", compute_metrics(ex_gt, ex_pred))

## 9. Evaluate selected test pages (checkpointed to Drive)

In [ ]:
RESULTS_CSV = OUTPUT_DIR / "per_image_results.csv"

if RESULTS_CSV.exists():
    previous = pd.read_csv(RESULTS_CSV).to_dict(orient="records")
    rows = [r for r in previous if r.get("sample_id") in {p["sample_id"] for p in pairs}]
    print(f"Resuming with {len(rows)} previously saved rows.")
else:
    rows = []

finished = {r["sample_id"] for r in rows}
for idx, sample in enumerate(tqdm(pairs, desc=f"OPRB eval ({MODEL_TAG})"), start=1):
    if sample["sample_id"] in finished:
        continue
    result = dict(sample)
    result["model_id"] = MODEL_ID
    try:
        prediction, elapsed, n_tokens, truncated = transcribe_page(sample["image_path"])
        ground_truth = " ".join(read_gt_tokens(sample["gt_path"]))
        result.update({
            "status": "ok",
            "prediction": prediction,
            "ground_truth": ground_truth,
            "latency_sec": elapsed,
            "generated_tokens": n_tokens,
            "possibly_truncated": truncated,
            **compute_metrics(ground_truth, prediction),
        })
    except Exception as exc:
        result.update({"status": "error", "error": repr(exc)})
        print("Failure:", sample["sample_id"], repr(exc))
    rows.append(result)
    if idx % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
results = pd.DataFrame(rows)
print("Saved:", RESULTS_CSV)
print("Successful:", (results.status == "ok").sum(), "/", len(results))

## 10. Summarize overall results and compare occlusion categories/types

In [ ]:
ok = results.loc[results["status"] == "ok"].copy()
assert len(ok), "No successful predictions."

metric_cols = ["cer", "wer", "token_precision", "token_recall", "token_f1", "latency_sec"]

def aggregate_metrics(df, group_by):
    return df.groupby(group_by, dropna=False).agg(
        n=("sample_id", "count"),
        **{f"mean_{c}": (c, "mean") for c in metric_cols},
        truncated=("possibly_truncated", "sum"),
    ).reset_index()

overall = {f"mean_{m}": float(ok[m].mean()) for m in metric_cols}
overall.update({
    "n_success": int(len(ok)),
    "n_errors": int(len(results) - len(ok)),
    "n_possibly_truncated": int(ok["possibly_truncated"].sum()),
})

by_category = aggregate_metrics(ok, ["category"])
by_type = aggregate_metrics(ok, ["category", "occlusion_type"])
by_density = aggregate_metrics(ok, ["category", "occlusion_density"])

print("OVERALL:", json.dumps(overall, indent=2))
display(by_category)
display(by_type)
display(by_density)

by_category.to_csv(OUTPUT_DIR / "summary_by_category.csv", index=False)
by_type.to_csv(OUTPUT_DIR / "summary_by_occlusion_type.csv", index=False)
by_density.to_csv(OUTPUT_DIR / "summary_by_density.csv", index=False)

## 11. Visualize errors

In [ ]:
ax = by_category.set_index("category")[["mean_cer", "mean_wer"]].plot(
    kind="bar", figsize=(7, 4), rot=0, title=f"{MODEL_TAG}: mean CER and WER by region"
)
ax.set_ylabel("Error rate (lower is better)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cer_wer_by_category.png", dpi=150)
plt.show()

ax = by_type.set_index("occlusion_type").groupby(level=0)["mean_wer"].mean().sort_values().plot(
    kind="barh", figsize=(9, 5), title=f"{MODEL_TAG}: mean WER by occlusion type"
)
ax.set_xlabel("Mean WER (lower is better)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wer_by_occlusion_type.png", dpi=150)
plt.show()

## 12. Export config and `.zip` results

In [ ]:
run_config = {
    "dataset": REPO_ID,
    "split": "test",
    "categories": CATEGORIES,
    "model_tag": MODEL_TAG,
    "model": MODEL_ID,
    "model_category": MODEL_CATEGORY,
    "precision": PRECISION_NOTE,
    "no_finetuning": True,
    "max_samples_per_category": MAX_SAMPLES_PER_CATEGORY,
    "random_seed": RANDOM_SEED,
    "normalize_case": NORMALIZE_CASE,
    "metrics": overall,
    "torch_version": torch.__version__,
    "hardware": GPU_NAME,
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
zip_name = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Results ZIP:", zip_name)